# DistilBERT - AG News Classification

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from mlflow_config import configure_mlflow
from tqdm import tqdm

configure_mlflow()
mlflow.set_experiment("AG_News_Classification")
os.makedirs("Figures", exist_ok=True)
os.makedirs("results", exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

/home/moussnaoui/PycharmProjects/MlopsProject_Text_Classification/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
GPU: NVIDIA A10


In [2]:
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")
train_df['text'] = train_df['Title'] + ' ' + train_df['Description']
test_df['text'] = test_df['Title'] + ' ' + test_df['Description']
train_texts = train_df['text'].values
test_texts = test_df['text'].values
y_train = (train_df['Class Index'] - 1).values
y_test = (test_df['Class Index'] - 1).values

In [3]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
max_len = 128
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=max_len)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True, max_length=max_len)

train_inputs = torch.tensor(train_encodings['input_ids'])
train_masks = torch.tensor(train_encodings['attention_mask'])
train_labels = torch.tensor(y_train)

test_inputs = torch.tensor(test_encodings['input_ids'])
test_masks = torch.tensor(test_encodings['attention_mask'])
test_labels = torch.tensor(y_test)

batch_size = 16
train_dataset = TensorDataset(train_inputs, train_masks, train_labels)
test_dataset = TensorDataset(test_inputs, test_masks, test_labels)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [4]:
with mlflow.start_run(run_name="DistilBERT"):
    mlflow.log_param("model_type", "DistilBERT")
    mlflow.log_param("model_name", "distilbert-base-uncased")
    mlflow.log_param("max_len", max_len)
    mlflow.log_param("batch_size", batch_size)
    
    model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=4)
    model.to(device)
    
    optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)
    epochs = 3
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("learning_rate", 2e-5)
    
    train_losses, val_accs = [], []
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            b_input_ids, b_labels = b_input_ids, b_labels = batch[0].to(device), batch[2].to(device)  # DistilBERT doesn't use token_type_ids
            b_input_mask = batch[1].to(device)
            model.zero_grad()
            outputs = model(b_input_ids, attention_mask=b_input_mask, labels=b_labels)
            loss = outputs.loss
            total_loss += loss.item()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
        train_losses.append(total_loss / len(train_loader))
        
        model.eval()
        preds, true_labels = [], []
        with torch.no_grad():
            for batch in test_loader:
                b_input_ids, b_input_mask, b_labels = tuple(t.to(device) for t in batch)
                outputs = model(b_input_ids, attention_mask=b_input_mask)
                preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
                true_labels.extend(b_labels.cpu().numpy())
        val_accs.append(accuracy_score(true_labels, preds))
        mlflow.log_metric(f"epoch_{epoch+1}_val_acc", val_accs[-1])
    
    y_pred = preds
    acc = accuracy_score(y_test, y_pred)
    mlflow.log_metric("test_accuracy", acc)
    
    report = classification_report(y_test, y_pred, target_names=['World', 'Sports', 'Business', 'Sci/Tech'])
    with open('results/DistilBERT_results.txt', 'w') as f:
        f.write(report)
        f.write(f'\nAccuracy: {acc}\n')
    mlflow.log_artifact('results/DistilBERT_results.txt')
    
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(range(1, epochs+1), val_accs, marker='o')
    plt.title('DistilBERT Validation Accuracy')
    plt.xlabel('Epoch')
    plt.subplot(1,2,2)
    plt.plot(range(1, epochs+1), train_losses, marker='o')
    plt.title('DistilBERT Training Loss')
    plt.xlabel('Epoch')
    plt.savefig('Figures/DistilBERT_accuracy_loss.png')
    plt.close()
    mlflow.log_artifact('Figures/DistilBERT_accuracy_loss.png')
    
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['World', 'Sports', 'Business', 'Sci/Tech'], yticklabels=['World', 'Sports', 'Business', 'Sci/Tech'])
    plt.title('DistilBERT Confusion Matrix')
    plt.savefig('Figures/DistilBERT_confusion_matrix.png')
    plt.close()
    mlflow.log_artifact('Figures/DistilBERT_confusion_matrix.png')
    print(report)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 775.07it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Epoch 3: 100%|██████████| 7500/7500 [08:33<00:00, 14.61it/s]


              precision    recall  f1-score   support

       World       0.96      0.95      0.96      1900
      Sports       0.99      0.99      0.99      1900
    Business       0.93      0.91      0.92      1900
    Sci/Tech       0.91      0.93      0.92      1900

    accuracy                           0.95      7600
   macro avg       0.95      0.95      0.95      7600
weighted avg       0.95      0.95      0.95      7600

